# Comparing marginal likelihood (evidence) estimators

This notebook compares a couple of (log) marginal likelihood estimators on two examples where an exact value is known, the beta-binomial model and Bayesian linear regression. 

In [ ]:
import jax
jax.config.update("jax_enable_x64", True)  
import jax.random as jrnd
import jax.numpy as jnp
import blackjax
import numpyro as npr
import numpyro.distributions as dist
import numpyro.distributions.transforms as nprb

import sys

import bamojax
from bamojax.base import Model
from bamojax.samplers import mcmc_sampler
from jax.scipy.special import betaln, gammaln

from bamojax.inference import MCMCInference, SMCInference
from bamojax.marginal_likelihoods.bridge_sampling import bridge_sampling
from bamojax.marginal_likelihoods.importance_sampling import importance_sampling
from bamojax.marginal_likelihoods.laplace import laplace_approximation
from bamojax.marginal_likelihoods.naive_monte_carlo import naive_monte_carlo
from bamojax.marginal_likelihoods.thames import thames


print('Python version:       ', sys.version)
print('Jax version:          ', jax.__version__)
print('BlackJax version:     ', blackjax.__version__)
print('Numpyro version:      ', npr.__version__)
print('BaMoJax version:      ', bamojax.__version__)
print('Jax default backend:  ', jax.default_backend())
print('Jax devices:          ', jax.devices())

Python version:        3.11.13 | packaged by conda-forge | (main, Jun  4 2025, 14:48:23) [GCC 13.3.0]
Jax version:           0.9.2
BlackJax version:      1.2.5
Numpyro version:       0.19.0
BaMoJax version:       0.3.14+0.gdf0aad8.dirty
Jax default backend:   cpu
Jax devices:           [CpuDevice(id=0)]


These analyses can take a while:

In [8]:
fast = False

Define models with exact log marginal likelihoods:

In [9]:
def beta_binomial(k=7, N=20, a=2.0, b=3.0):
    model = Model("beta-binomial")
    theta = model.add_node("theta", distribution=dist.Beta(a, b))
    model.add_node("k", distribution=dist.Binomial, parents=dict(probs=theta, total_count=N), observations=k)
    exact = (gammaln(N + 1) - gammaln(k + 1) - gammaln(N - k + 1)) + betaln(k + a, N - k + b) - betaln(a, b)
    return model, exact

#
NOISE, PRIOR_SD = 0.5, 2.0

def linear_regression(X, y, name):
    p = X.shape[1]
    model = Model(name)
    w = model.add_node("w", distribution=dist.Normal(jnp.zeros(p), PRIOR_SD * jnp.ones(p)))
    x_node = model.add_node("X", observations=X)
    model.add_node("y", distribution=dist.Normal, parents=dict(w=w, X=x_node),
                   link_fn=lambda w, X: dict(loc=X @ w, scale=NOISE), observations=y)
    cov = NOISE**2 * jnp.eye(X.shape[0]) + PRIOR_SD**2 * X @ X.T
    exact = dist.MultivariateNormal(jnp.zeros(X.shape[0]), cov).log_prob(y)
    return model, exact

#

In [10]:
def nuts_posterior(model, key, num_samples):
    m = model.get_model_size()
    kernel = mcmc_sampler(model, mcmc_kernel=blackjax.nuts,
                          mcmc_parameters=dict(step_size=0.5, inverse_mass_matrix=jnp.eye(m)))  # overridden by warmup
    engine = MCMCInference(model, mcmc_kernel=kernel, num_samples=num_samples, num_burn=0, num_warmup=500)
    return engine.run(key)["states"]

#
def smc_evidence(model, key, num_particles, num_mutations=50):
    m = model.get_model_size()
    kernel = mcmc_sampler(model, mcmc_kernel=blackjax.normal_random_walk, mcmc_parameters=dict(sigma=0.1 * jnp.eye(m)))
    engine = SMCInference(model, mcmc_kernel=kernel, num_particles=num_particles, num_mutations=num_mutations)
    return engine.run(key)["lml"]

#
def all_estimates(model, key, is_proposal, bridge_bijectors, laplace_kwargs, sizes):
    keys = jrnd.split(key, 6)
    posterior = nuts_posterior(model, keys[0], sizes["posterior"])
    return {
        "naive Monte Carlo": naive_monte_carlo(keys[1], model, num_prior_draws=sizes["nmc"], num_chunks=5, pb=False),
        "importance sampling": importance_sampling(keys[2], model, g_IS=is_proposal, num_samples=sizes["is"]),
        "Laplace": laplace_approximation(keys[3], model, **laplace_kwargs),
        "SMC": smc_evidence(model, keys[4], sizes["smc"][0], num_mutations=sizes["smc"][1]),
        "bridge sampling": bridge_sampling(keys[5], model, posterior, bijectors=dict(bridge_bijectors))[0],
        "THAMES": thames(keys[5], model, posterior),
    }

#
def print_table(title, exact, estimates):
    print(f"\n{title}\n" + "-" * len(title))
    print(f"{'estimator':<22}{'log p(D)':>12}{'error':>10}")
    print(f"{'exact':<22}{float(exact):>12.4f}")
    for name, est in estimates.items():
        print(f"{name:<22}{float(est):>12.4f}{float(est - exact):>+10.4f}")

#

Evaluate on beta-binomial model:

In [11]:
sizes = dict(posterior=4_000, nmc=4_000, **{"is": 10_000}, smc=(1_000, 50)) if fast else \
        dict(posterior=20_000, nmc=40_000, **{"is": 50_000}, smc=(5_000, 100))
key = jrnd.PRNGKey(1234)

model, exact = beta_binomial()
key, sub = jrnd.split(key)
est = all_estimates(model, sub,
                    is_proposal={"theta": dist.Beta(4.5, 8.0)},          # a bit wider than Beta(9, 16)
                    bridge_bijectors={"theta": nprb.SigmoidTransform()},  # map (0, 1) -> R for the proposal
                    laplace_kwargs=dict(bounds=({"theta": 1e-6}, {"theta": 1 - 1e-6})),
                    sizes=sizes)
print_table("Beta-binomial (k=7, N=20, Beta(2, 3) prior)", exact, est)

Adapting NUTS HMC parameters... 

W1002 18:11:24.803546 4131072 pjrt_executable.cc:642] Assume version compatibility. PjRt-IFRT does not track XLA executable versions.
E1002 18:11:24.876259 4131643 cpu_aot_loader.cc:220] Loading XLA:CPU AOT result. Target machine feature +prefer-no-gather is not  supported on the host machine. Machine type used for XLA:CPU compilation doesn't match the machine type for execution. Compile machine features: [+64bit,+adx,+aes,+avx,+avx2,+avx512bw,+avx512cd,+avx512dq,+avx512f,+avx512vl,+avx512vnni,+bmi,+bmi2,+clflushopt,+clwb,+cmov,+crc32,+cx16,+cx8,+f16c,+fma,+fsgsbase,+fxsr,+invpcid,+lzcnt,+mmx,+movbe,+pclmul,+pku,+popcnt,+prefer-no-gather,+prefer-no-scatter,+prfchw,+rdrnd,+rdseed,+sahf,+sse,+sse2,+sse3,+sse4.1,+sse4.2,+ssse3,+xsave,+xsavec,+xsaveopt,+xsaves,-amx-avx512,-amx-bf16,-amx-complex,-amx-fp16,-amx-fp8,-amx-int8,-amx-movrs,-amx-tf32,-amx-tile,-avx10.1,-avx10.2,-avx512bf16,-avx512bitalg,-avx512fp16,-avx512ifma,-avx512vbmi,-avx512vbmi2,-avx512vp2intersect,-avx512vpopcntdq,-avxifma

done.

Beta-binomial (k=7, N=20, Beta(2, 3) prior)
-------------------------------------------
estimator                 log p(D)     error
exact                      -2.5377
naive Monte Carlo          -2.5372   +0.0004
importance sampling        -2.5378   -0.0001
Laplace                    -2.5074   +0.0302
SMC                        -2.5347   +0.0030
bridge sampling            -2.5377   -0.0001
THAMES                     -2.5399   -0.0022


Bayesian linear regression:

In [12]:
key, kx, ky = jrnd.split(key, 3)
X = jrnd.normal(kx, (40, 3))
y = X @ jnp.array([1.0, -2.0, 0.5]) + NOISE * jrnd.normal(ky, (40,))
results = {}
for name, cols in [("full", [0, 1, 2]), ("reduced", [0, 2])]:
    model, exact = linear_regression(X[:, cols], y, name)
    key, sub = jrnd.split(key)
    p = len(cols)
    est = all_estimates(model, sub,
                        is_proposal={"w": dist.StudentT(5., jnp.linalg.lstsq(X[:, cols], y)[0], 0.3 * jnp.ones(p))},
                        bridge_bijectors={}, laplace_kwargs={}, sizes=sizes)
    print_table(f"Linear regression, {name} model ({p} predictors)", exact, est)
    results[name] = (exact, est)

exact_bf = results["full"][0] - results["reduced"][0]
print(f"\nlog Bayes factor full vs reduced: exact {float(exact_bf):.3f}")
for name in results["full"][1]:
    log_bf = results["full"][1][name] - results["reduced"][1][name]
    print(f"  {name:<22}{float(log_bf):>10.3f}")


Adapting NUTS HMC parameters... 

W1002 18:11:57.073633 4131072 pjrt_executable.cc:642] Assume version compatibility. PjRt-IFRT does not track XLA executable versions.
E1002 18:11:57.117573 4131643 cpu_aot_loader.cc:220] Loading XLA:CPU AOT result. Target machine feature +prefer-no-gather is not  supported on the host machine. Machine type used for XLA:CPU compilation doesn't match the machine type for execution. Compile machine features: [+64bit,+adx,+aes,+avx,+avx2,+avx512bw,+avx512cd,+avx512dq,+avx512f,+avx512vl,+avx512vnni,+bmi,+bmi2,+clflushopt,+clwb,+cmov,+crc32,+cx16,+cx8,+f16c,+fma,+fsgsbase,+fxsr,+invpcid,+lzcnt,+mmx,+movbe,+pclmul,+pku,+popcnt,+prefer-no-gather,+prefer-no-scatter,+prfchw,+rdrnd,+rdseed,+sahf,+sse,+sse2,+sse3,+sse4.1,+sse4.2,+ssse3,+xsave,+xsavec,+xsaveopt,+xsaves,-amx-avx512,-amx-bf16,-amx-complex,-amx-fp16,-amx-fp8,-amx-int8,-amx-movrs,-amx-tf32,-amx-tile,-avx10.1,-avx10.2,-avx512bf16,-avx512bitalg,-avx512fp16,-avx512ifma,-avx512vbmi,-avx512vbmi2,-avx512vp2intersect,-avx512vpopcntdq,-avxifma

done.

Linear regression, full model (3 predictors)
--------------------------------------------
estimator                 log p(D)     error
exact                     -40.4479
naive Monte Carlo         -40.5765   -0.1285
importance sampling       -40.4140   +0.0340
Laplace                   -40.4479   -0.0000
SMC                       -40.4833   -0.0353
bridge sampling           -40.4480   -0.0001
THAMES                    -40.4412   +0.0067
Adapting NUTS HMC parameters... 

W1002 18:12:27.074076 4131072 pjrt_executable.cc:642] Assume version compatibility. PjRt-IFRT does not track XLA executable versions.
E1002 18:12:27.117509 4131643 cpu_aot_loader.cc:220] Loading XLA:CPU AOT result. Target machine feature +prefer-no-gather is not  supported on the host machine. Machine type used for XLA:CPU compilation doesn't match the machine type for execution. Compile machine features: [+64bit,+adx,+aes,+avx,+avx2,+avx512bw,+avx512cd,+avx512dq,+avx512f,+avx512vl,+avx512vnni,+bmi,+bmi2,+clflushopt,+clwb,+cmov,+crc32,+cx16,+cx8,+f16c,+fma,+fsgsbase,+fxsr,+invpcid,+lzcnt,+mmx,+movbe,+pclmul,+pku,+popcnt,+prefer-no-gather,+prefer-no-scatter,+prfchw,+rdrnd,+rdseed,+sahf,+sse,+sse2,+sse3,+sse4.1,+sse4.2,+ssse3,+xsave,+xsavec,+xsaveopt,+xsaves,-amx-avx512,-amx-bf16,-amx-complex,-amx-fp16,-amx-fp8,-amx-int8,-amx-movrs,-amx-tf32,-amx-tile,-avx10.1,-avx10.2,-avx512bf16,-avx512bitalg,-avx512fp16,-avx512ifma,-avx512vbmi,-avx512vbmi2,-avx512vp2intersect,-avx512vpopcntdq,-avxifma

done.

Linear regression, reduced model (2 predictors)
-----------------------------------------------
estimator                 log p(D)     error
exact                    -231.2378
naive Monte Carlo        -231.1892   +0.0486
importance sampling      -231.2416   -0.0038
Laplace                  -231.2378   +0.0000
SMC                      -231.2327   +0.0051
bridge sampling          -231.2380   -0.0002
THAMES                   -231.2104   +0.0274

log Bayes factor full vs reduced: exact 190.790
  naive Monte Carlo        190.613
  importance sampling      190.828
  Laplace                  190.790
  SMC                      190.749
  bridge sampling          190.790
  THAMES                   190.769
